# GPT-SoVITS em português do Brasil: treinar a SUA voz e narrar slides (Colab, GPU grátis)

Este notebook faz o GPT-SoVITS falar **pt-BR com a sua voz**:

1. instala o GPT-SoVITS e aplica o **patch pt-BR** do Narrador (frontend de texto português → fonemas);
2. você envia **5 a 20 minutos** da sua voz lendo texto em português (celular serve; sem música);
3. treina o SoVITS (timbre) e o GPT (pronúncia/ritmo) com esse áudio: ~20-40 min na T4;
4. sobe o servidor `api_v2` com os seus pesos e abre a **interface web** do Narrador para gerar o MP4.

Sem treino (zero-shot) o modelo fala português com sotaque forte e erros; **o treino é o que faz funcionar bem**.
Antes de começar: *Ambiente de execução → Alterar o tipo de ambiente → GPU (T4)*.

In [ ]:
#@title 1. Instalar GPT-SoVITS + patch pt-BR + Narrador (8-12 min)
%cd /content
!git clone -q --depth 1 https://github.com/RVC-Boss/GPT-SoVITS.git
!git clone -q https://github.com/mscpatricio/OBRAS-SAUDE.git /content/repo
%cd /content/GPT-SoVITS
!apt-get -qq install -y ffmpeg libreoffice-impress > /dev/null
!pip install -q -r requirements.txt 2>&1 | tail -1
!pip install -q huggingface_hub torchcodec resampy /content/repo/narrador[web,online] chatterbox-tts 2>&1 | tail -1
from huggingface_hub import snapshot_download
snapshot_download("lj1995/GPT-SoVITS", local_dir="GPT_SoVITS/pretrained_models",
                  allow_patterns=["s1v3.ckpt", "v2Pro/*", "sv/*", "chinese-roberta-wwm-ext-large/*", "chinese-hubert-base/*"])
!wget -q https://huggingface.co/XXXXRT/GPT-SoVITS-Pretrained/resolve/main/nltk_data.zip && unzip -q -o nltk_data.zip -d /usr/local && rm nltk_data.zip
!cp /content/repo/narrador/narrador/gptsovits_ptbr/*.py /content/GPT-SoVITS/
!python aplicar_patch.py /content/GPT-SoVITS
print("Instalado e com suporte a pt-BR")

In [ ]:
#@title 2. Enviar a gravação da sua voz (5-20 min em português; pode ser um ou vários arquivos)
from google.colab import files
import pathlib, shutil
pathlib.Path("/content/minha_voz").mkdir(exist_ok=True)
up = files.upload()
for n in up:
    shutil.move(n, f"/content/minha_voz/{n}")
print("Arquivos:", sorted(p.name for p in pathlib.Path("/content/minha_voz").iterdir()))

In [ ]:
#@title 3. Treinar a sua voz em pt-BR (20-40 min)
nome = "minha_voz" #@param {type:"string"}
epocas_sovits = 8 #@param {type:"integer"}
epocas_gpt = 15 #@param {type:"integer"}
%cd /content/GPT-SoVITS
!python treinar_ptbr.py --audio /content/minha_voz --nome {nome} --epocas-sovits {epocas_sovits} --epocas-gpt {epocas_gpt} --whisper large-v3-turbo

In [ ]:
#@title 4. Subir o servidor com os seus pesos e abrir a interface web do Narrador
import glob, os, subprocess, time, requests, yaml
nome = "minha_voz" #@param {type:"string"}
sov = sorted(glob.glob(f"/content/GPT-SoVITS/SoVITS_weights_v2Pro/{nome}_e*.pth"), key=os.path.getmtime)[-1]
gpt = sorted(glob.glob(f"/content/GPT-SoVITS/GPT_weights_v2Pro/{nome}-e*.ckpt"), key=os.path.getmtime)[-1]
print("SoVITS:", sov); print("GPT:", gpt)
cfg_path = "/content/GPT-SoVITS/GPT_SoVITS/configs/tts_infer.yaml"
cfg = yaml.safe_load(open(cfg_path))
cfg["custom"] = {"version": "v2Pro", "device": "cuda", "is_half": True, "t2s_weights_path": gpt, "vits_weights_path": sov,
                 "bert_base_path": "GPT_SoVITS/pretrained_models/chinese-roberta-wwm-ext-large",
                 "cnhuhbert_base_path": "GPT_SoVITS/pretrained_models/chinese-hubert-base"}
yaml.safe_dump(cfg, open(cfg_path, "w"))
log = open("/content/api_v2.log", "w")
subprocess.Popen(["python", "api_v2.py", "-a", "127.0.0.1", "-p", "9880", "-c", cfg_path], cwd="/content/GPT-SoVITS", stdout=log, stderr=subprocess.STDOUT)
for _ in range(120):
    try:
        requests.get("http://127.0.0.1:9880/control", params={"command": ""}, timeout=2); break
    except Exception:
        time.sleep(2)
print("Servidor no ar. Abrindo a interface web (use o link público abaixo).")
#@markdown Na interface: aba 1 envie 3-10 s da sua voz como referência (com a transcrição), aba 2 teste, aba 3 gere o vídeo.
!cd /content && narrador web --compartilhar --host 0.0.0.0 --saida /content/saida --motor gpt-sovits -o text_lang=pt -o url=http://127.0.0.1:9880

In [ ]:
#@title 5. (Alternativa sem interface) gerar o vídeo direto pela linha de comando
#@markdown Envie antes o .pptx (e opcionalmente o roteiro .md) para /content/entrada.
from google.colab import files
import pathlib, shutil, glob
pathlib.Path("/content/entrada").mkdir(exist_ok=True)
for n in files.upload():
    shutil.move(n, f"/content/entrada/{n}")
apres = [p for p in glob.glob("/content/entrada/*") if p.endswith((".pptx", ".pdf"))][0]
!cd /content && narrador voz preparar "{sorted(glob.glob('/content/minha_voz/*'))[0]}" --saida /content/voz/ref.wav --duracao 9
!cd /content && narrador gerar "{apres}" --motor gpt-sovits --voz /content/voz/ref.wav -o text_lang=pt -o url=http://127.0.0.1:9880 --legendas --saida /content/saida
files.download(glob.glob("/content/saida/*/*.mp4")[-1])